# UPCT Medical Image Segmentation Challenge 2026-27
## Práctica 6

**Asignatura:** Procesado de Imágenes Médicas (521104007)

**Profesor:** Juan Zapata

> **Cómo funciona esta serie de notebooks:** cada semana se libera un notebook nuevo (Práctica 6 a 10) con el contenido de esa sesión. **Este primer notebook (P6) es la base de tu proyecto**: complétalo, y a partir de la próxima semana copia las celdas nuevas de cada práctica y pégalas **al final de este mismo notebook** — no empieces uno nuevo cada semana ni pegues de nuevo prácticas ya hechas. Al llegar a la P10 tendrás tu propio notebook completo con las 5 prácticas: ese es el que entregarás en el aula virtual.
>
> Cada práctica se abre en un **runtime nuevo** de Colab, así que cada día tendrás que ejecutar todas las celdas de tu notebook desde el principio (incluidas las de semanas anteriores) — no hace falta rehacer nada: los checkpoints en Drive detectan que ya existen y se cargan directamente en vez de reentrenar.

## Guía de Sesiones (2 horas por sesión)
| Práctica | Fechas (Grupo A / B) | Objetivo de la Sesión | Checkpoint Visual |
|----------|----------------------|-----------------------|-------------------|
| ▶ **P6** | 28 Oct - 2 Nov | EDA, Dataset y formato RLE | 6 imágenes con máscaras + RLE OK |
| **P7** | 9-11 Nov | Baseline U-Net y 1ª Submission | Gráficas de Loss + Submission Kaggle |
| **P8** | 16-18 Nov | Data Augmentation y mejora | Comparativa Baseline vs Augmented |
| **P9** | 23-25 Nov | Inferencia, Threshold y Errores | 5 imágenes normales + 2 casos de error |
| **P10** | 30 Nov-2 Dic | TTA, Submission Final y Defensa | Mejor Dice Score + Defensa Oral |

> **Regla de Oro:** Según el Art. 7.5 del Reglamento de Evaluación UPCT, la asistencia y validación del Checkpoint en el aula es obligatoria para superar la práctica.


## Descargar el Dataset de la Competición

El dataset de este reto es **privado** y solo está disponible dentro de la competición en Kaggle.

### Instrucciones:
1. **Acepta la invitación** al reto que te llegó por email (o accede directamente):
   [UPCT Medical Image Segmentation Challenge](https://www.kaggle.com/competitions/upct-medical-image-segmentation-challenge-2026-27)

2. Dentro de la competición, ve a la pestaña **"Data"** (en el menú superior).

3. Haz clic en **"Download All"** o descarga el archivo **`upct-medical-image-segmentation-challenge-2026-27.zip`** (aprox. 150 MB).

4. **Sube el archivo ZIP** a tu Google Drive en una carpeta llamada **`PIM_Challenge`**:
   - Entra en [Google Drive](https://drive.google.com)
   - Crea la carpeta `PIM_Challenge` (si no existe)
   - Sube el ZIP ahí

> **IMPORTANTE:**
> - **NO cambies el nombre del archivo ZIP** (debe mantener el nombre original)
> - **NO descomprimas el ZIP** antes de subirlo (lo haremos desde Colab)
> - Si no puedes acceder a la competición, contacta con el profesor: juan.zapata@upct.es

## Estructura del dataset

Una vez descomprimido, el dataset tendrá esta estructura:


In [ ]:
# ============================================================
# VERIFICACIÓN Y EXPLORACIÓN DEL DATASET DESCARGADO
# ============================================================

# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Instalar tree para visualizar carpetas
!apt-get install -y tree > /dev/null 2>&1

# 3. Localizar el ZIP en PIM_Challenge
import os
import zipfile
import shutil
from pathlib import Path

DRIVE_FOLDER = '/content/drive/MyDrive/PIM_Challenge'
EXTRACT_DIR = Path('/content/PIM_Challenge_dataset')

print(f"\n Buscando dataset en: {DRIVE_FOLDER}\n")

# Listar contenido de la carpeta
if os.path.exists(DRIVE_FOLDER):
    print(" Contenido de PIM_Challenge:")
    for f in os.listdir(DRIVE_FOLDER):
        size = os.path.getsize(os.path.join(DRIVE_FOLDER, f)) / (1024*1024)
        print(f"   - {f} ({size:.1f} MB)")
else:
    print(f" La carpeta {DRIVE_FOLDER} no existe")
    raise FileNotFoundError(DRIVE_FOLDER)

# Buscar el ZIP
zip_files = [f for f in os.listdir(DRIVE_FOLDER) if f.endswith('.zip')]

if not zip_files:
    print("\n No hay ningún archivo ZIP en PIM_Challenge")
    print(" Sube el ZIP que descargaste de Kaggle a esa carpeta")
    raise FileNotFoundError("No hay ZIP en PIM_Challenge")

ZIP_PATH = os.path.join(DRIVE_FOLDER, zip_files[0])
print(f"\n ZIP encontrado: {zip_files[0]}")

# 4. Descomprimir
if not EXTRACT_DIR.exists():
    print(f"\n Descomprimiendo {zip_files[0]}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print("Descompresión completada.\n")
else:
    print(f"\nDataset ya descomprimido en {EXTRACT_DIR}\n")

# 5. Reorganizar si es necesario (Kaggle a veces descomprime en /content/ directamente)
print("Verificando estructura del dataset...")

if not (EXTRACT_DIR / 'train').exists() and Path('/content/train').exists():
    print(" Dataset descomprimido en /content/. Reorganizando...")

    if not EXTRACT_DIR.exists():
        EXTRACT_DIR.mkdir()

    items_to_move = ['train', 'test', 'sample_submission.csv']
    for item in items_to_move:
        src = Path(f'/content/{item}')
        dst = EXTRACT_DIR / item

        if src.exists():
            if dst.exists():
                if dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    dst.unlink()

            shutil.move(str(src), str(dst))
            print(f"    Movido: {item}")
        else:
            print(f"    No encontrado: {item}")

    print(" Reorganización completada.\n")
elif (EXTRACT_DIR / 'train').exists():
    print(" Estructura correcta detectada.\n")
else:
    print(" No se encuentra la estructura esperada. Buscando carpetas train/test...\n")

# 6. Mostrar estructura del dataset
print("="*60)
print(" ESTRUCTURA DEL DATASET")
print("="*60)
!tree -L 3 /content/PIM_Challenge_dataset --dirsfirst

# 7. Estadísticas detalladas
print("\n" + "="*60)
print(" ESTADÍSTICAS DETALLADAS")
print("="*60)

import glob

train_imgs = glob.glob(f'{EXTRACT_DIR}/train/images/*.png')
train_masks = glob.glob(f'{EXTRACT_DIR}/train/masks/*.png')
test_imgs = glob.glob(f'{EXTRACT_DIR}/test/images/*.png')

# Contar por clase
train_benign = len([f for f in train_imgs if 'benign' in f])
train_malignant = len([f for f in train_imgs if 'malignant' in f])
train_normal = len([f for f in train_imgs if 'normal' in f])

test_benign = len([f for f in test_imgs if 'benign' in f])
test_malignant = len([f for f in test_imgs if 'malignant' in f])
test_normal = len([f for f in test_imgs if 'normal' in f])

print(f"\n TRAIN SET: {len(train_imgs)} imágenes")
print(f"   ├─ Benign:     {train_benign:3d} imágenes")
print(f"   ├─ Malignant:  {train_malignant:3d} imágenes")
print(f"   └─ Normal:     {train_normal:3d} imágenes")
print(f"    Máscaras:   {len(train_masks)} archivos")

print(f"\n TEST SET: {len(test_imgs)} imágenes (SIN máscaras)")
print(f"   ├─ Benign:     {test_benign:3d} imágenes")
print(f"   ├─ Malignant:  {test_malignant:3d} imágenes")
print(f"   └─ Normal:     {test_normal:3d} imágenes")

has_csv = os.path.exists(f'{EXTRACT_DIR}/sample_submission.csv')
print(f"\nsample_submission.csv: {' Existe' if has_csv else ' No existe'}")

# 8. Verificación de integridad
print("\n" + "="*60)
print(" VERIFICACIÓN DE INTEGRIDAD")
print("="*60)

checks = [
    (len(train_imgs) == 546, f"Train images: {len(train_imgs)}/546"),
    (len(train_masks) == 546, f"Train masks: {len(train_masks)}/546"),
    (len(test_imgs) == 234, f"Test images: {len(test_imgs)}/234"),
    (has_csv, "sample_submission.csv existe"),
]

all_ok = True
for ok, desc in checks:
    print(f"   {'OK' if ok else 'noOK'} {desc}")
    if not ok:
        all_ok = False

# Verificación final
if all_ok:
    print("\n ¡Dataset correcto y listo para trabajar!")
else:
    print("\n Hay problemas en el dataset. Revisa los archivos.")

# PRÁCTICA 6: Exploración del Dataset y Formato RLE
## Sesión única (28 Oct - miércoles - y 2 Nov - lunes -)

### Objetivos de la sesión:
1. Entender la estructura del dataset BUSI (benign, malignant, normal)
2. Visualizar imágenes y máscaras de segmentación
3. Comprender el formato RLE (Run-Length Encoding) para Kaggle
4. Crear el DataLoader para entrenamiento

### Contexto clínico
El dataset BUSI contiene 780 imágenes de ultrasonido mamario:
- **Benign (437)**: Lesiones no cancerosas
- **Malignant (210)**: Lesiones cancerosas
- **Normal (133)**: Tejido sano (sin lesión)

**Reto clínico**: El modelo debe saber decir "no hay tumor" en las imágenes normales.

> **CHECKPOINT P6:** Mostrar al profesor la visualización de imágenes con sus máscaras superpuestas y la verificación de que tus funciones `mask_to_rle`/`rle_to_mask` reconstruyen la máscara original correctamente.

### Tiempo estimado: 2 horas

## Bloque 6.1: Segmentación Semántica en Imágenes Médicas
### Clasificación vs Detección vs Segmentación

| Tarea | Pregunta que responde | Salida |
|-------|----------------------|--------|
| **Clasificación** | ¿Hay tumor? | Etiqueta: "benigno" / "maligno" |
| **Detección** | ¿Dónde está el tumor? | Bounding box |
| **Segmentación** | ¿Qué píxeles son tumor? | Máscara píxel a píxel |

### ¿Por qué segmentación y no clasificación?
- En medicina, el **tamaño y forma** del tumor importan (estadiaje)
- Permite calcular **volumen**, **bordes irregulares**, **invasión**
- Es la base de la **radiómica** y el diagnóstico asistido

### El reto específico: Ultrasonido mamario (BUS)
- Imágenes **ruidosas** y de **bajo contraste**
- Tumores **pequeños** (a veces < 5% de la imagen)
- Artefactos: sombras acústicas, reverberaciones
- **Clase "normal"**: el modelo debe aprender a NO detectar nada

## Tarea 6.1: Carga y estructuración del Dataset
Antes de entrenar ningún modelo, necesitamos organizar la información en memoria.
Vamos a usar la librería **Pandas** para crear un DataFrame que actúe como índice de nuestro dataset, relacionando cada imagen con su clase y su máscara.

### Instrucciones:
1. Define la ruta base del dataset: `/content/PIM_Challenge_dataset`.
2. Recorre la carpeta `train/images` buscando todas las imágenes `.png`.
3. Para cada imagen, extrae su **clase** (`benign`, `malignant` o `normal`) basándote en el nombre de directorio.
4. Busca la ruta de su máscara correspondiente en la carpeta `train/masks`. *(Pista: el nombre de la máscara suele ser el mismo que la imagen pero añadiendo `_mask` al final).*
5. Guarda toda esta información en un **DataFrame de Pandas** con las columnas: `class`, `image_path`, `mask_path` y `filename`.
6. Muestra por pantalla el número total de imágenes y la distribución de clases (cuántas hay de cada tipo: benign, malignat y normal ).

> **Librerías recomendadas:** `pandas`, `pathlib`, `os`.


In [ ]:
# ============================================================
# TAREA 6.1: CARGA DEL DATASET EN PANDAS
# ============================================================

DATA_DIR = Path('/content/PIM_Challenge_dataset')

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Crea una lista vacía para almacenar los datos
# 2. Itera sobre las clases: ['benign', 'malignant', 'normal']
# 3. Para cada clase, busca las imágenes en DATA_DIR / 'train' / 'images'
# 4. Construye la ruta de la máscara correspondiente
# 5. Añade un diccionario con la información a la lista
# 6. Convierte la lista en un DataFrame

data = []

# Tu código aquí...


df_train = pd.DataFrame(data)

print(f"Dataset cargado: {len(df_train)} imágenes de entrenamiento.")
print("\n Distribución de clases en Train:")
print(df_train['class'].value_counts())

## Bloque 6.2: El Dataset BUSI (Breast Ultrasound Images)
### Origen y composición
- Publicado por **Al-Dhabyani et al. (2020)**
- 780 imágenes de ultrasonido mamario
- 3 clases:
  - **Benign (437)**: fibroadenomas, quistes...
  - **Malignant (210)**: carcinomas
  - **Normal (133)**: tejido sano

### El desbalance de clases

>  **Pregunta para pensar:** ¿Por qué hay más imágenes benignas que malignas en la realidad clínica?

### El "Reto Oculto": las imágenes normales
La mayoría de competiciones de segmentación asumen que **todas las imágenes tienen el objeto a segmentar**. En este reto:
- Las imágenes **normal** tienen máscara **vacía** (toda negra)
- Si tu modelo pinta un tumor donde no lo hay → **FALSO POSITIVO**
- Consecuencia clínica: biopsia innecesaria, ansiedad, coste sanitario

### Métricas: ¿por qué no usamos Accuracy?
Imagina un modelo que dice "no hay tumor" en TODAS las imágenes:
- Accuracy: ~85% (porque el 85% de los píxeles son fondo)
- Dice Score: **0** (no ha detectado ningún tumor)

Por eso en segmentación médica usamos **Dice Score** o **IoU**.

> **Pregunta para pensar:** "Si tuvierais que diagnosticar a una paciente, ¿qué os preocuparía más: un falso positivo o un falso negativo? ¿Por qué?"


## Tarea 6.2: Visualización y el "Reto Clínico Oculto"
No podemos confiar en los datos a ciegas. Vamos a visualizar los datos para asegurarnos de que las máscaras están alineadas con las imágenes y entender la dificultad clínica del reto.

### Instrucciones:
1. Selecciona **1 imagen aleatoria de cada clase** (benign, malignant, normal) de tu DataFrame.
2. Crea una figura de **3 filas x 3 columnas** usando `matplotlib` (una fila por clase).
3. Para cada imagen, carga y muestra en las 3 columnas:
   * **Columna 1 (Original):** La imagen en RGB. *(Recuerda que `cv2.imread` la lee en BGR, tendrás que convertirla).*
   * **Columna 2 (Máscara Real):** La máscara en escala de grises.
   * **Columna 3 (Overlay):** Copia la imagen original y **pinta de ROJO `[255, 0, 0]`** los píxeles donde la máscara indica que hay tumor.
4. Analiza las imágenes de la clase **"normal"**. Escribe un `print()` al final explicando:
   * ¿Qué ves en sus máscaras?
   * ¿Qué pasaría clínicamente si tu modelo de IA predice un tumor (píxeles rojos) en una de estas imágenes?

> **CHECKPOINT P6.1:** Cuando tengas la figura 3x3 generada, llama al profesor para que valide tu visualización y tu conclusión clínica.

> **Librerías recomendadas:** `cv2`, `matplotlib.pyplot`, `numpy`.

In [ ]:
# ============================================================
# TAREA 6.2: VISUALIZACIÓN DE EJEMPLOS
# ============================================================

# ESCRIBE TU CÓDIGO AQUÍ

# 1. Selecciona 1 imagen aleatoria de cada clase
# TU CODIGO AQUÍ

# 2. Crea la figura
# TU CODIGO AQUÍ

# 3. Itera sobre las muestras y muestra las 3 columnas
# TU CODIGO AQUÍ

# plt.tight_layout()
# plt.show()

# 4. Análisis de imágenes normales
# TU CODIGO AQUÍ

# Comprueba si las máscaras de las imágenes normales están vacías
# TU CODIGO AQUÍ

# print(f"Imágenes 'normal' con máscara totalmente negra (vacía): {empty_masks}/{len(normal_samples)}")
# print("\n CONCLUSIÓN CLÍNICA:")
# print("Si tu modelo predice un tumor en una imagen 'normal', estará cometiendo un FALSO POSITIVO.")
# print("En medicina, esto puede llevar a biopsias innecesarias y ansiedad en la paciente.")
# print("Tu modelo debe aprender a NO segmentar nada cuando no hay lesión.")

## Bloque 6.3: Formatos de Submission y Métricas
### ¿Por qué RLE y no PNG?
Kaggle necesita un formato **texto** para:
1. **Compresión**: una máscara de 512x512 = 262,144 píxeles → en RLE pueden ser solo 20 números
2. **Validación automática**: fácil de parsear y comparar
3. **Múltiples objetos**: se pueden codificar varios tumores en una imagen

### Métricas clave en Kaggle

**Dice Score (F1 Score):**
$$Dice = \frac{2 \cdot |A \cap B|}{|A| + |B|}$$

**IoU (Intersection over Union):**
$$IoU = \frac{|A \cap B|}{|A \cup B|}$$

**Relación:** $Dice = \frac{2 \cdot IoU}{1 + IoU}$

| Dice Score | Interpretación clínica |
|------------|----------------------|
| 0.9 - 1.0 | Segmentación excelente |
| 0.7 - 0.9 | Buena (típico en competiciones) |
| 0.5 - 0.7 | Aceptable, mejorable |
| < 0.5 | Modelo no útil clínicamente |

### Calcular manualmente:
"Calculad a mano el Dice de este ejemplo:"

    Predicción: 100 píxeles de tumor
    Ground truth: 100 píxeles de tumor
    Intersección: 80 píxeles
    Respuesta: Dice = 2·80/(100+100) = 0.80

## Tarea 6.3: El Formato RLE (Run-Length Encoding)
Kaggle no acepta máscaras como imágenes PNG para las submissions. En su lugar, exige que las máscaras se envíen en formato de texto usando **Run-Length Encoding (RLE)**.

### ¿Qué es RLE?
Es un método de compresión muy simple: en lugar de guardar cada píxel, guardamos **secuencias de píxeles iguales**.

**Ejemplo:**
Imagina una máscara de 5x5 (25 píxeles) donde los píxeles del centro son tumor (1) y el resto es fondo (0):

0 0 0 0 0

0 1 1 1 0

0 1 1 1 0

0 1 1 1 0

0 0 0 0 0


En formato RLE, esto se expresa como:
`"7 3 12 3 17 3"`

**¿Qué significa?**
- `7 3` → Empezando en el píxel 7, hay 3 píxeles de tumor
- `12 3` → Empezando en el píxel 12, hay 3 píxeles de tumor
- `17 3` → Empezando en el píxel 17, hay 3 píxeles de tumor

> **IMPORTANTE:** Kaggle lee las imágenes **por columnas** (de arriba a abajo, luego siguiente columna), no por filas. Y los índices empiezan en **1**, no en 0.

### Instrucciones:
1. Implementa la función `mask_to_rle(mask)` que:
   - Reciba una máscara binaria (numpy array de 0s y 1s)
   - La aplane por columnas (usa `.T.flatten()`)
   - Detecte los cambios de 0 a 1 y de 1 a 0
   - Devuelva un string con el formato RLE (pares de números separados por espacios)

2. Implementa la función `rle_to_mask(rle_string, height, width)` que:
   - Reciba un string RLE y las dimensiones de la imagen
   - Reconstruya la máscara binaria original
   - Devuelva un numpy array de la forma `(height, width)`

3. **Prueba tus funciones:**
   - Crea una máscara de prueba (puedes usar una máscara real del dataset o crear una artificial)
   - Conviértela a RLE con `mask_to_rle()`
   - Reconstrúyela con `rle_to_mask()`
   - Verifica que la máscara original y la reconstruida son **idénticas** (usa `np.array_equal()`)

4. **Visualiza el resultado:**
   - Muestra 3 imágenes: máscara original, máscara reconstruida, y la diferencia (debe ser toda negra si funciona bien)

> **Pista:** Para detectar los cambios en la secuencia de píxeles, puedes usar `np.where(pixels[1:] != pixels[:-1])`.

> **CHECKPOINT P6.2:** Muestra al profesor que tus funciones RLE funcionan correctamente (la diferencia entre máscara original y reconstruida debe ser cero).

In [ ]:
# ============================================================
# TAREA 6.3: IMPLEMENTACIÓN DE RLE
# ============================================================

import numpy as np

# ESCRIBE TU CÓDIGO AQUÍ

def mask_to_rle(mask):
    """
    Convierte una máscara binaria a formato RLE.

    Args:
        mask: numpy array binario (0s y 1s) de forma (height, width)

    Returns:
        str: string con el formato RLE (pares de números separados por espacios)
    """
    # Tu código aquí
    pass


def rle_to_mask(rle_string, height, width):
    """
    Convierte un string RLE a una máscara binaria.

    Args:
        rle_string: string con formato RLE
        height: altura de la imagen
        width: ancho de la imagen

    Returns:
        numpy array binario de forma (height, width)
    """
    # Tu código aquí
    pass


# ============================================================
# PRUEBA DE LAS FUNCIONES (no modificar)
# ============================================================

# Cargar una máscara real del dataset
sample_mask_path = df_train[df_train['class'] == 'benign'].iloc[0]['mask_path']
original_mask = cv2.imread(sample_mask_path, cv2.IMREAD_GRAYSCALE)
original_mask_bin = (original_mask > 127).astype(np.uint8)

print(f"Máscara original: forma {original_mask_bin.shape}, píxeles de tumor: {original_mask_bin.sum()}")

# Convertir a RLE
rle_encoded = mask_to_rle(original_mask_bin)
print(f"RLE encoded (primeros 50 caracteres): {rle_encoded[:50]}...")

# Reconstruir desde RLE
reconstructed_mask = rle_to_mask(rle_encoded, original_mask_bin.shape[0], original_mask_bin.shape[1])
print(f"Máscara reconstruida: forma {reconstructed_mask.shape}, píxeles de tumor: {reconstructed_mask.sum()}")

# Verificar que son idénticas
are_equal = np.array_equal(original_mask_bin, reconstructed_mask)
print(f"\n¿Son idénticas? {are_equal}")

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_mask_bin, cmap='gray')
axes[0].set_title("Máscara Original")
axes[0].axis('off')

axes[1].imshow(reconstructed_mask, cmap='gray')
axes[1].set_title("Reconstruida desde RLE")
axes[1].axis('off')

difference = np.abs(original_mask_bin.astype(int) - reconstructed_mask.astype(int))
axes[2].imshow(difference, cmap='hot')
axes[2].set_title(f"Diferencia (debe ser negra)\nSuma de diferencias: {difference.sum()}")
axes[2].axis('off')

plt.tight_layout()
plt.show()

if are_equal:
    print("\n¡Funciona perfectamente! Tus funciones RLE son correctas.")
else:
    print("\nHay diferencias. Revisa tu implementación.")